# NFL Big Data Bowl 2026  
## 03 – Feature Engineering v2 (Yards Gained Modeling Dataset)

In this notebook, we construct and document movement- and context-based features
derived from player tracking and supplementary data. These features are designed
to support predictive modeling of play effectiveness, measured by yards gained,
and are used to answer Research Questions 2 and 3.

The focus of this notebook is feature definition, computation, and justification.
Model training and evaluation are deferred to the next notebook.

## Motivation

The exploratory analysis in RQ1 demonstrated that separation between the targeted
receiver and defenders is strongly associated with pass completion. However,
separation alone does not fully explain how successful a play is in terms of yards
gained.

To predict yards gained, we require a richer representation of the play that
captures:
- offensive intent (route design),
- defensive intent (coverage schemes),
- spatial relationships between players,
- and movement dynamics such as speed, acceleration, and pursuit angles.

This notebook defines a structured set of features that combine contextual and
movement-based information into a play-level dataset suitable for machine learning.

**Goal:** Build a single play-level dataset (one row per targeted-receiver play) to predict:

- **Target:** `yards_gained`

**Key design choices:**
- We only use data available **before the play outcome is known** (no leakage).
- We represent each play using:
  1) **Throw-snapshot geometry** (receiver + defenders at a reference frame)
  2) **Pre-throw dynamics** aggregated across **all input frames**
  3) **Contextual play fields** (route, coverage, down/distance, etc.)

Outputs:
- `data/model_table_yards.parquet`
- `data/model_table_yards.csv`
- `data/model_table_yards_data_dictionary.csv`

Model training and evaluation are performed in the next notebook.

In [1]:
# Imports & paths

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent  # if notebook is in /notebooks
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

PROJECT_ROOT: /Users/siamsadman/Git_Projects/git-projects/statistical_consulting_project/nfl_big_data_bowl_2026/nfl-yards-gained-prediction
DATA_DIR: /Users/siamsadman/Git_Projects/git-projects/statistical_consulting_project/nfl_big_data_bowl_2026/nfl-yards-gained-prediction/data


## Loading Raw Tracking and Supplementary Data

We load two datasets:

1. **Tracking input data**  
   Frame-level player movement data available *before the throw*.

2. **Supplementary play-level data**  
   Contextual information and outcomes, including the modeling target
   `yards_gained`.

These datasets are later combined into a single modeling table.

In [2]:
from src.data_loading import load_merged_input_with_supp, load_supplementary

# Raw merged tracking (input frames) + supplementary join
input_with_supp = load_merged_input_with_supp()

# Play-level contextual + outcomes
supp_df = load_supplementary()

print("input_with_supp:", input_with_supp.shape)
print("supp_df:", supp_df.shape)

input_with_supp.head()

Loading: input_2023_w01.csv
Loading: input_2023_w02.csv
Loading: input_2023_w03.csv
Loading: input_2023_w04.csv
Loading: input_2023_w05.csv
Loading: input_2023_w06.csv
Loading: input_2023_w07.csv
Loading: input_2023_w08.csv
Loading: input_2023_w09.csv
Loading: input_2023_w10.csv
Loading: input_2023_w11.csv
Loading: input_2023_w12.csv
Loading: input_2023_w13.csv
Loading: input_2023_w14.csv
Loading: input_2023_w15.csv
Loading: input_2023_w16.csv
Loading: input_2023_w17.csv
Loading: input_2023_w18.csv
Tracking input shape: (4880579, 23)
Supplementary shape: (18009, 41)


/Users/siamsadman/Git_Projects/git-projects/statistical_consulting_project/nfl_big_data_bowl_2026/nfl-yards-gained-prediction/src/data_loading.py:34: DtypeWarning: Columns (0: play_action) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(SUPPLEMENTARY_PATH)


Merged dataset shape: (4880579, 62)
Supplementary shape: (18009, 41)
input_with_supp: (4880579, 62)
supp_df: (18009, 41)


/Users/siamsadman/Git_Projects/git-projects/statistical_consulting_project/nfl_big_data_bowl_2026/nfl-yards-gained-prediction/src/data_loading.py:34: DtypeWarning: Columns (0: play_action) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(SUPPLEMENTARY_PATH)


,game_id,play_id,player_to_predict,nfl_id,frame_id,play_direction,absolute_yardline_number,player_name,player_height,player_weight,...,team_coverage_type,penalty_yards,pre_penalty_yards_gained,yards_gained,expected_points,expected_points_added,pre_snap_home_team_win_probability,pre_snap_visitor_team_win_probability,home_team_win_probability_added,visitor_team_win_probility_added
0,2023090700,101,False,54527,1,right,42,Bryan Cook,6-1,210,...,COVER_2_ZONE,NaN,0,0,0.927021,-2.145443,0.590426,0.409574,0.04972,-0.04972
1,2023090700,101,False,54527,2,right,42,Bryan Cook,6-1,210,...,COVER_2_ZONE,NaN,0,0,0.927021,-2.145443,0.590426,0.409574,0.04972,-0.04972
2,2023090700,101,False,54527,3,right,42,Bryan Cook,6-1,210,...,COVER_2_ZONE,NaN,0,0,0.927021,-2.145443,0.590426,0.409574,0.04972,-0.04972
3,2023090700,101,False,54527,4,right,42,Bryan Cook,6-1,210,...,COVER_2_ZONE,NaN,0,0,0.927021,-2.145443,0.590426,0.409574,0.04972,-0.04972
4,2023090700,101,False,54527,5,right,42,Bryan Cook,6-1,210,...,COVER_2_ZONE,NaN,0,0,0.927021,-2.145443,0.590426,0.409574,0.04972,-0.04972


## Data Sanity Checks

Before feature engineering, we verify:
- required columns are present,
- play identifiers align across datasets,
- the number of unique plays is consistent.

This step helps catch data issues early.

In [3]:
required_tracking_cols = [
    "game_id", "play_id", "frame_id",
    "player_role", "player_side",
    "nfl_id", "x", "y"
]
missing = [c for c in required_tracking_cols if c not in input_with_supp.columns]
print("Missing tracking cols:", missing)

required_supp_cols = ["game_id", "play_id", "yards_gained"]
missing2 = [c for c in required_supp_cols if c not in supp_df.columns]
print("Missing supplementary cols:", missing2)

print("Unique plays in input:", input_with_supp[["game_id", "play_id"]].drop_duplicates().shape[0])
print("Unique plays in supp:", supp_df[["game_id", "play_id"]].drop_duplicates().shape[0])

Missing tracking cols: []
Missing supplementary cols: []
Unique plays in input: 14108
Unique plays in supp: 18009


## Helper Functions for Geometry and Angles

We define helper functions to compute:
- Euclidean distance,
- wrapped angular differences,
- unit direction vectors,
- safe numerical divisions.

These functions are reused throughout the feature engineering process.

In [4]:
def euclid(x1, y1, x2, y2):
    return np.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def wrap_angle_deg(angle_deg):
    """Wrap any angle in degrees to [-180, 180]."""
    return (angle_deg + 180) % 360 - 180

def angle_to_unit_vector_deg(theta_deg):
    """Convert direction angle in degrees into a 2D unit vector."""
    th = np.deg2rad(theta_deg)
    return np.cos(th), np.sin(th)

def safe_div(a, b, eps=1e-9):
    return a / (b + eps)

## Defining the Reference Frame (Snapshot Moment)

To compute spatial features consistently, we define a **reference frame**:

- the **last available input frame** for the targeted receiver.

This frame is the closest moment to the ball being released and ensures that no post-throw information is used.

In [5]:
# Identify targeted receiver rows
wr_df = input_with_supp[input_with_supp["player_role"] == "Targeted Receiver"].copy()

# Reference snapshot frame per play (WR)
ref = (
    wr_df.groupby(["game_id", "play_id", "nfl_id"], as_index=False)["frame_id"]
    .max()
    .rename(columns={"nfl_id": "nfl_id_wr", "frame_id": "ref_frame_id"})
)

print("Reference frames table:", ref.shape)
ref.head()

Reference frames table: (14108, 4)


,game_id,play_id,nfl_id_wr,ref_frame_id
0,2023090700,101,44930,26
1,2023090700,194,41325,32
2,2023090700,219,53591,17
3,2023090700,361,38696,51
4,2023090700,436,53541,20


## Targeted Receiver (WR) Snapshot Features

At the reference frame, we extract the targeted receiver’s:
- position,
- speed,
- acceleration,
- movement direction.

These features describe the receiver’s state at the moment of the throw.

In [6]:
wr_snapshot = (
    wr_df.merge(
        ref,
        left_on=["game_id", "play_id", "nfl_id", "frame_id"],
        right_on=["game_id", "play_id", "nfl_id_wr", "ref_frame_id"],
        how="inner"
    )
    .copy()
)

wr_snapshot = wr_snapshot.rename(columns={
    "x": "wr_x",
    "y": "wr_y",
    "s": "wr_s",
    "a": "wr_a",
    "dir": "wr_dir",
    "o": "wr_o",
    "player_position": "wr_position",
    "player_name": "wr_name",
})

keep_cols = ["game_id", "play_id", "nfl_id_wr", "ref_frame_id", "wr_x", "wr_y"]
for c in ["wr_s", "wr_a", "wr_dir", "wr_o", "wr_position"]:
    if c in wr_snapshot.columns:
        keep_cols.append(c)

for c in ["ball_land_x", "ball_land_y", "play_direction", "absolute_yardline_number"]:
    if c in wr_snapshot.columns and c not in keep_cols:
        keep_cols.append(c)

wr_snapshot = wr_snapshot[keep_cols].drop_duplicates()

print("wr_snapshot:", wr_snapshot.shape)
wr_snapshot.head()

wr_snapshot: (14108, 15)


,game_id,play_id,nfl_id_wr,ref_frame_id,wr_x,wr_y,wr_s,wr_a,wr_dir,wr_o,wr_position,ball_land_x,ball_land_y,play_direction,absolute_yardline_number
0,2023090700,101,44930,26,52.43,14.14,7.90,2.68,99.25,106.80,WR,63.259998,-0.22,right,42
1,2023090700,194,41325,32,88.98,22.23,6.09,2.14,245.74,314.63,RB,84.940002,21.75,left,89
2,2023090700,219,53591,17,75.98,10.22,3.85,2.77,268.92,343.54,TE,75.849998,11.49,left,79
3,2023090700,361,38696,51,34.19,48.33,1.51,3.87,18.99,194.89,WR,26.100000,49.18,right,22
4,2023090700,436,53541,20,33.67,37.80,3.59,6.08,116.90,136.35,WR,34.889999,34.82,right,31


## Defender Positions at the Reference Frame

We extract all defenders present at the same reference frame as the WR. This allows us to analyze:
- defensive pressure,
- coverage structure,
- spatial relationships between the WR and defenders.

In [7]:
def_df = input_with_supp[input_with_supp["player_side"] == "Defense"].copy()

def_at_ref = def_df.merge(ref, on=["game_id", "play_id"], how="inner")
def_at_ref = def_at_ref[def_at_ref["frame_id"] == def_at_ref["ref_frame_id"]].copy()

def_at_ref = def_at_ref.rename(columns={
    "nfl_id": "nfl_id_def",
    "x": "def_x",
    "y": "def_y",
    "s": "def_s",
    "a": "def_a",
    "dir": "def_dir",
    "o": "def_o",
    "player_position": "def_position"
})

keep_def = ["game_id", "play_id", "nfl_id_def", "ref_frame_id", "def_x", "def_y"]
for c in ["def_s", "def_a", "def_dir", "def_o", "def_position"]:
    if c in def_at_ref.columns:
        keep_def.append(c)

def_at_ref = def_at_ref[keep_def]

print("def_at_ref:", def_at_ref.shape)
def_at_ref.head()

def_at_ref: (94293, 11)


,game_id,play_id,nfl_id_def,ref_frame_id,def_x,def_y,def_s,def_a,def_dir,def_o,def_position
25,2023090700,101,54527,26,57.23,35.59,2.97,3.42,133.26,251.72,FS
51,2023090700,101,46137,26,55.82,17.67,5.34,1.80,134.17,184.99,SS
77,2023090700,101,52546,26,48.01,12.44,2.93,4.75,192.18,309.47,CB
103,2023090700,101,53487,26,49.18,22.00,5.03,2.27,133.98,182.20,MLB
129,2023090700,101,54486,26,49.58,41.96,4.22,3.53,88.36,144.07,CB


## Identifying the Top-K Nearest Defenders

For each play, we:
1. compute distances from the WR to all defenders,
2. rank defenders by distance,
3. retain the K nearest defenders (K = 5).

This avoids losing information from secondary defenders while keeping the feature space manageable.

In [8]:
K = 5

tmp = def_at_ref.merge(
    wr_snapshot[
        ["game_id", "play_id", "nfl_id_wr", "ref_frame_id", "wr_x", "wr_y"]
        + ([c for c in ["wr_s", "wr_dir"] if c in wr_snapshot.columns])
    ],
    on=["game_id", "play_id", "ref_frame_id"],
    how="inner"
)

tmp["dx"] = tmp["wr_x"] - tmp["def_x"]
tmp["dy"] = tmp["wr_y"] - tmp["def_y"]
tmp["dist"] = np.sqrt(tmp["dx"]**2 + tmp["dy"]**2)

tmp["def_rank"] = tmp.groupby(["game_id", "play_id", "nfl_id_wr"])["dist"].rank(method="first")

tmp_sorted = tmp.sort_values(["game_id", "play_id", "nfl_id_wr", "def_rank"]).copy()
tmp_topk = tmp_sorted[tmp_sorted["def_rank"] <= K].copy()

print("tmp_topk:", tmp_topk.shape)
tmp_topk.head()

tmp_topk: (70356, 20)


,game_id,play_id,nfl_id_def,ref_frame_id,def_x,def_y,def_s,def_a,def_dir,def_o,def_position,nfl_id_wr,wr_x,wr_y,wr_s,wr_dir,dx,dy,dist,def_rank
2,2023090700,101,52546,26,48.01,12.44,2.93,4.75,192.18,309.47,CB,44930,52.43,14.14,7.9,99.25,4.42,1.70,4.735652,1.0
1,2023090700,101,46137,26,55.82,17.67,5.34,1.80,134.17,184.99,SS,44930,52.43,14.14,7.9,99.25,-3.39,-3.53,4.894180,2.0
3,2023090700,101,53487,26,49.18,22.00,5.03,2.27,133.98,182.20,MLB,44930,52.43,14.14,7.9,99.25,3.25,-7.86,8.505416,3.0
0,2023090700,101,54527,26,57.23,35.59,2.97,3.42,133.26,251.72,FS,44930,52.43,14.14,7.9,99.25,-4.80,-21.45,21.980503,4.0
4,2023090700,101,54486,26,49.58,41.96,4.22,3.53,88.36,144.07,CB,44930,52.43,14.14,7.9,99.25,2.85,-27.82,27.965602,5.0


## Defender Direction and Pursuit Features

For each of the top-K defenders, we compute:
- angular difference relative to the WR’s direction,
- an approach score indicating whether the defender is moving toward the WR,
- projected closing speed.

These features capture *how defenders are moving*, not just where they are.

In [9]:
if "def_dir" in tmp_topk.columns and "wr_dir" in tmp_topk.columns:
    tmp_topk["angle_diff_def_wr"] = wrap_angle_deg(tmp_topk["def_dir"] - tmp_topk["wr_dir"])
else:
    tmp_topk["angle_diff_def_wr"] = np.nan

if "def_dir" in tmp_topk.columns:
    ux, uy = angle_to_unit_vector_deg(tmp_topk["def_dir"].fillna(0).values)
    tmp_topk["def_unit_vx"] = ux
    tmp_topk["def_unit_vy"] = uy

    r_norm = np.sqrt(tmp_topk["dx"]**2 + tmp_topk["dy"]**2)
    tmp_topk["approach_score"] = safe_div(
        tmp_topk["dx"] * tmp_topk["def_unit_vx"] + tmp_topk["dy"] * tmp_topk["def_unit_vy"],
        r_norm,
    )
else:
    tmp_topk["approach_score"] = np.nan

if "def_s" in tmp_topk.columns:
    tmp_topk["def_speed_toward_wr"] = tmp_topk["def_s"] * tmp_topk["approach_score"]
else:
    tmp_topk["def_speed_toward_wr"] = np.nan

tmp_topk[
    ["game_id", "play_id", "nfl_id_wr", "def_rank", "dist", "dx", "dy",
     "angle_diff_def_wr", "approach_score", "def_speed_toward_wr"]
].head()

,game_id,play_id,nfl_id_wr,def_rank,dist,dx,dy,angle_diff_def_wr,approach_score,def_speed_toward_wr
2,2023090700,101,44930,1.0,4.735652,4.42,1.70,92.93,-0.988074,-2.895058
1,2023090700,101,44930,2.0,4.894180,-3.39,-3.53,34.92,-0.034708,-0.185339
3,2023090700,101,44930,3.0,8.505416,3.25,-7.86,34.73,-0.930318,-4.679499
0,2023090700,101,44930,4.0,21.980503,-4.80,-21.45,34.01,-0.561020,-1.666230
4,2023090700,101,44930,5.0,27.965602,2.85,-27.82,-10.89,-0.991469,-4.184001


## Pivoting Top-K Defender Features

To create a machine-learning-friendly table, we pivot top-K defender features into wide format:
- `dist_def_1`, `dist_def_2`, ...
- `dx_def_1`, `dy_def_1`, ...
- `approach_def_1`, ...

Each play now occupies a single row.

In [10]:
def pivot_topk(df, value_col, prefix):
    wide = (
        df.pivot_table(
            index=["game_id", "play_id", "nfl_id_wr", "ref_frame_id"],
            columns="def_rank",
            values=value_col,
            aggfunc="first"
        )
        .rename(columns=lambda r: f"{prefix}{int(r)}")
        .reset_index()
    )
    return wide

wide_dist = pivot_topk(tmp_topk, "dist", "dist_def_")
wide_dx = pivot_topk(tmp_topk, "dx", "dx_def_")
wide_dy = pivot_topk(tmp_topk, "dy", "dy_def_")
wide_ang = pivot_topk(tmp_topk, "angle_diff_def_wr", "angdiff_def_")
wide_app = pivot_topk(tmp_topk, "approach_score", "approach_def_")
wide_clo = pivot_topk(tmp_topk, "def_speed_toward_wr", "closev_def_")

topk_wide = wide_dist.merge(wide_dx, on=["game_id", "play_id", "nfl_id_wr", "ref_frame_id"], how="left")
topk_wide = topk_wide.merge(wide_dy, on=["game_id", "play_id", "nfl_id_wr", "ref_frame_id"], how="left")
topk_wide = topk_wide.merge(wide_ang, on=["game_id", "play_id", "nfl_id_wr", "ref_frame_id"], how="left")
topk_wide = topk_wide.merge(wide_app, on=["game_id", "play_id", "nfl_id_wr", "ref_frame_id"], how="left")
topk_wide = topk_wide.merge(wide_clo, on=["game_id", "play_id", "nfl_id_wr", "ref_frame_id"], how="left")

print("topk_wide:", topk_wide.shape)
topk_wide.head()

topk_wide: (14107, 34)


def_rank,game_id,play_id,nfl_id_wr,ref_frame_id,dist_def_1,dist_def_2,dist_def_3,dist_def_4,dist_def_5,dx_def_1,...,approach_def_1,approach_def_2,approach_def_3,approach_def_4,approach_def_5,closev_def_1,closev_def_2,closev_def_3,closev_def_4,closev_def_5
0,2023090700,101,44930,26,4.735652,4.894180,8.505416,21.980503,27.965602,4.42,...,-0.988074,-0.034708,-0.930318,-0.561020,-0.991469,-2.895058,-0.185339,-4.679499,-1.666230,-4.184001
1,2023090700,194,41325,32,1.883215,9.704478,12.557838,12.582230,19.375234,-1.81,...,0.776695,-0.477555,-0.061067,0.996960,0.240700,4.264057,-1.499524,-0.280910,6.211063,0.269584
2,2023090700,219,53591,17,4.906832,7.192781,11.222584,13.611671,21.522047,4.81,...,-0.443564,0.492223,0.379221,0.978740,0.973391,-1.268594,1.678480,1.236259,1.644283,2.199863
3,2023090700,361,38696,51,2.180138,18.205606,21.368973,30.474017,36.150024,-1.47,...,-0.285646,0.258380,-0.552652,0.988441,-0.105906,-0.265650,1.671721,-1.912177,5.031166,-0.505173
4,2023090700,436,53541,20,2.872786,8.933695,10.408967,20.931815,21.478978,-2.48,...,-0.588687,-0.445979,-0.534815,-0.789249,0.250048,-0.435628,-1.476190,-2.053689,-2.778155,1.005193


## Summary Features over All Defenders

Beyond individual defenders, we compute summary statistics over *all defenders*:
- mean and variability of distances,
- distance percentiles,
- counts of defenders within fixed radii.

These features describe overall defensive density.

In [11]:
all_def = tmp_sorted.copy()

thresholds = [1, 2, 3, 5, 8]
for t in thresholds:
    all_def[f"def_within_{t}"] = (all_def["dist"] <= t).astype(int)

summary = (
    all_def.groupby(["game_id", "play_id", "nfl_id_wr", "ref_frame_id"])
    .agg(
        dist_def_min=("dist", "min"),
        dist_def_mean=("dist", "mean"),
        dist_def_std=("dist", "std"),
        dist_def_p10=("dist", lambda x: np.nanpercentile(x, 10)),
        dist_def_p25=("dist", lambda x: np.nanpercentile(x, 25)),
        dist_def_p50=("dist", lambda x: np.nanpercentile(x, 50)),
        dist_def_p75=("dist", lambda x: np.nanpercentile(x, 75)),
        dist_def_p90=("dist", lambda x: np.nanpercentile(x, 90)),
        **{f"n_def_within_{t}": (f"def_within_{t}", "sum") for t in thresholds}
    )
    .reset_index()
)

print("summary:", summary.shape)
summary.head()

summary: (14107, 17)


,game_id,play_id,nfl_id_wr,ref_frame_id,dist_def_min,dist_def_mean,dist_def_std,dist_def_p10,dist_def_p25,dist_def_p50,dist_def_p75,dist_def_p90,n_def_within_1,n_def_within_2,n_def_within_3,n_def_within_5,n_def_within_8
0,2023090700,101,44930,26,4.735652,13.616271,10.687899,4.799063,4.894180,8.505416,21.980503,25.571562,0,0,0,2,2
1,2023090700,194,41325,32,1.883215,14.781918,8.013034,6.575973,11.131158,12.582230,20.852528,23.414137,0,1,1,1,1
2,2023090700,219,53591,17,4.906832,16.887142,10.602307,6.278401,9.207683,13.611671,23.602762,29.038326,0,0,0,1,2
3,2023090700,361,38696,51,2.180138,24.325399,13.339774,10.192872,18.996448,25.921495,34.731022,36.861829,0,0,1,1,1
4,2023090700,436,53541,20,2.872786,15.767967,10.030146,5.903240,9.302513,15.670391,21.342187,25.730270,0,0,1,1,1


## Pre-Throw Wide Receiver Movement Dynamics

Using all input frames, we compute WR movement summaries such as:
- total path length,
- average and maximum speed,
- acceleration and direction variability.

This captures how the WR moves leading up to the throw.

In [12]:
wr_frames = wr_df.rename(columns={"nfl_id": "nfl_id_wr"}).copy()

wr_frames = wr_frames.sort_values(["game_id", "play_id", "nfl_id_wr", "frame_id"])
wr_frames["wr_dx_step"] = wr_frames.groupby(["game_id", "play_id", "nfl_id_wr"])["x"].diff()
wr_frames["wr_dy_step"] = wr_frames.groupby(["game_id", "play_id", "nfl_id_wr"])["y"].diff()
wr_frames["wr_step_dist"] = np.sqrt(wr_frames["wr_dx_step"]**2 + wr_frames["wr_dy_step"]**2)

agg_wr = (
    wr_frames.groupby(["game_id", "play_id", "nfl_id_wr"])
    .agg(
        n_frames=("frame_id", "count"),
        wr_path_length=("wr_step_dist", "sum"),
        wr_x_start=("x", "first"),
        wr_y_start=("y", "first"),
        wr_x_end=("x", "last"),
        wr_y_end=("y", "last"),
    )
    .reset_index()
)

for col in ["s", "a", "dir"]:
    if col in wr_frames.columns:
        agg = (
            wr_frames.groupby(["game_id", "play_id", "nfl_id_wr"])[col]
            .agg(["mean", "max", "std"])
            .reset_index()
            .rename(columns={
                "mean": f"wr_{col}_mean",
                "max": f"wr_{col}_max",
                "std": f"wr_{col}_std"
            })
        )
        agg_wr = agg_wr.merge(agg, on=["game_id", "play_id", "nfl_id_wr"], how="left")

print("agg_wr:", agg_wr.shape)
agg_wr.head()

agg_wr: (14108, 18)


,game_id,play_id,nfl_id_wr,n_frames,wr_path_length,wr_x_start,wr_y_start,wr_x_end,wr_y_end,wr_s_mean,wr_s_max,wr_s_std,wr_a_mean,wr_a_max,wr_a_std,wr_dir_mean,wr_dir_max,wr_dir_std
0,2023090700,101,44930,26,11.773586,41.03,12.17,52.43,14.14,4.611154,7.90,2.936111,2.975769,5.56,1.502068,79.829231,156.35,21.470646
1,2023090700,194,41325,32,8.491335,93.33,27.85,88.98,22.23,2.723750,6.13,2.350042,2.350938,4.58,1.371663,173.902813,359.46,81.838252
2,2023090700,219,53591,17,4.620993,80.57,10.65,75.98,10.22,2.780000,4.69,1.812340,2.742941,5.21,1.685140,249.373529,269.57,51.175275
3,2023090700,361,38696,51,17.888180,20.65,37.90,34.19,48.33,3.500980,6.78,2.358838,2.765490,5.71,1.455840,65.777059,158.93,32.963307
4,2023090700,436,53541,20,4.892001,29.46,35.92,33.67,37.80,2.512000,4.43,1.666558,3.491500,6.25,1.984565,71.476000,196.22,36.128909


## Separation Dynamics Across Frames

For each frame, separation is defined as the distance to the nearest defender.

Across all frames we compute:
- minimum separation,
- mean separation,
- variability,
- fraction of frames under tight coverage.

These features summarize how difficult it was for the WR to get open.

In [13]:
def_frames = def_df.copy()

wr_f = wr_frames[["game_id", "play_id", "frame_id", "nfl_id_wr", "x", "y"]].rename(
    columns={"x": "wr_x", "y": "wr_y"}
)
df_f = def_frames[["game_id", "play_id", "frame_id", "nfl_id", "x", "y"]].rename(
    columns={"nfl_id": "nfl_id_def", "x": "def_x", "y": "def_y"}
)

joined = df_f.merge(wr_f, on=["game_id", "play_id", "frame_id"], how="inner")
joined["dist"] = np.sqrt((joined["wr_x"] - joined["def_x"])**2 + (joined["wr_y"] - joined["def_y"])**2)

nearest = (
    joined.sort_values(["game_id", "play_id", "nfl_id_wr", "frame_id", "dist"])
    .groupby(["game_id", "play_id", "nfl_id_wr", "frame_id"], as_index=False)
    .first()
)

nearest = nearest.rename(columns={"dist": "sep_frame"})

sep_agg = (
    nearest.groupby(["game_id", "play_id", "nfl_id_wr"])
    .agg(
        sep_min=("sep_frame", "min"),
        sep_mean=("sep_frame", "mean"),
        sep_std=("sep_frame", "std"),
        sep_p25=("sep_frame", lambda x: np.nanpercentile(x, 25)),
        sep_p50=("sep_frame", lambda x: np.nanpercentile(x, 50)),
        sep_p75=("sep_frame", lambda x: np.nanpercentile(x, 75)),
    )
    .reset_index()
)

for t in [1, 2, 3]:
    frac = (
        nearest.assign(flag=(nearest["sep_frame"] < t).astype(int))
        .groupby(["game_id", "play_id", "nfl_id_wr"])["flag"]
        .mean()
        .reset_index()
        .rename(columns={"flag": f"frac_sep_below_{t}"})
    )
    sep_agg = sep_agg.merge(frac, on=["game_id", "play_id", "nfl_id_wr"], how="left")

print("sep_agg:", sep_agg.shape)
sep_agg.head()

sep_agg: (14107, 12)


,game_id,play_id,nfl_id_wr,sep_min,sep_mean,sep_std,sep_p25,sep_p50,sep_p75,frac_sep_below_1,frac_sep_below_2,frac_sep_below_3
0,2023090700,101,44930,1.066771,2.770382,1.011555,1.872267,3.073829,3.559012,0.0,0.269231,0.461538
1,2023090700,194,41325,1.001798,4.189085,2.509043,1.736266,4.102659,6.671739,0.0,0.343750,0.406250
2,2023090700,219,53591,4.906832,5.927040,0.637236,5.363283,6.049876,6.554121,0.0,0.000000,0.000000
3,2023090700,361,38696,2.143105,6.057146,3.590496,2.667735,4.921839,9.967827,0.0,0.000000,0.352941
4,2023090700,436,53541,2.872786,4.648680,1.019285,3.785929,4.910604,5.648982,0.0,0.000000,0.050000


## Separation at the Throw Snapshot

We extract the nearest-defender separation specifically at the reference frame, which serves as the snapshot closest to the release moment.

In [14]:
sep_at_throw = nearest.merge(
    ref,
    on=["game_id", "play_id", "nfl_id_wr"],
    how="inner"
)

sep_at_throw = sep_at_throw[sep_at_throw["frame_id"] == sep_at_throw["ref_frame_id"]].copy()

sep_at_throw = (
    sep_at_throw[["game_id", "play_id", "nfl_id_wr", "sep_frame"]]
    .rename(columns={"sep_frame": "sep_at_throw"})
    .drop_duplicates()
)

print("sep_at_throw:", sep_at_throw.shape)
sep_at_throw.head()

sep_at_throw: (14107, 4)


,game_id,play_id,nfl_id_wr,sep_at_throw
25,2023090700,101,44930,4.735652
57,2023090700,194,41325,1.883215
74,2023090700,219,53591,4.906832
125,2023090700,361,38696,2.180138
145,2023090700,436,53541,2.872786


## Adding Contextual Play Features

We join play-level context from the supplementary dataset, including:
- down and distance,
- route of the targeted receiver,
- coverage scheme,
- field position.

These variables provide tactical context.

In [15]:
context_cols = [
    "season", "week", "down", "yards_to_go", "absolute_yardline_number",
    "pass_length", "offense_formation", "receiver_alignment",
    "team_coverage_man_zone", "team_coverage_type",
    "play_action", "dropback_type", "dropback_distance",
    "pass_location_type", "defenders_in_the_box",
    "route_of_targeted_receiver",
    "pass_result",
    "yards_gained",
]

context_keep = ["game_id", "play_id"] + [c for c in context_cols if c in supp_df.columns]
context = supp_df[context_keep].drop_duplicates(["game_id", "play_id"]).copy()

print("context:", context.shape)
context.head()

context: (18009, 19)


,game_id,play_id,season,week,down,yards_to_go,pass_length,offense_formation,receiver_alignment,team_coverage_man_zone,team_coverage_type,play_action,dropback_type,dropback_distance,pass_location_type,defenders_in_the_box,route_of_targeted_receiver,pass_result,yards_gained
0,2023090700,3461,2023,1,3,12,18,EMPTY,3x2,ZONE_COVERAGE,COVER_2_ZONE,False,TRADITIONAL,5.30,INSIDE_BOX,6,IN,C,18
1,2023090700,461,2023,1,1,10,13,SINGLEBACK,3x1,ZONE_COVERAGE,COVER_6_ZONE,True,TRADITIONAL,4.72,INSIDE_BOX,7,POST,C,21
2,2023090700,1940,2023,1,2,10,18,SHOTGUN,3x1,ZONE_COVERAGE,COVER_2_ZONE,False,TRADITIONAL,4.44,INSIDE_BOX,6,OUT,I,0
3,2023090700,1711,2023,1,1,10,23,SHOTGUN,3x1,ZONE_COVERAGE,COVER_2_ZONE,False,TRADITIONAL,5.36,INSIDE_BOX,5,CORNER,C,26
4,2023090700,1588,2023,1,1,10,38,SHOTGUN,2x2,ZONE_COVERAGE,COVER_4_ZONE,False,TRADITIONAL,4.59,INSIDE_BOX,6,POST,I,0


## Assembling the Final Modeling Table

All feature blocks are merged into a single table with:
- one row per play,
- hundreds of engineered features,
- target variable `yards_gained`.

In [16]:
model = wr_snapshot.merge(context, on=["game_id", "play_id"], how="left")
model = model.merge(topk_wide, on=["game_id", "play_id", "nfl_id_wr", "ref_frame_id"], how="left")
model = model.merge(summary, on=["game_id", "play_id", "nfl_id_wr", "ref_frame_id"], how="left")
model = model.merge(agg_wr, on=["game_id", "play_id", "nfl_id_wr"], how="left")
model = model.merge(sep_agg, on=["game_id", "play_id", "nfl_id_wr"], how="left")
model = model.merge(sep_at_throw, on=["game_id", "play_id", "nfl_id_wr"], how="left")

print("Final model table:", model.shape)
model.head()

Final model table: (14108, 100)


,game_id,play_id,nfl_id_wr,ref_frame_id,wr_x,wr_y,wr_s,wr_a,wr_dir,wr_o,...,sep_min,sep_mean,sep_std,sep_p25,sep_p50,sep_p75,frac_sep_below_1,frac_sep_below_2,frac_sep_below_3,sep_at_throw
0,2023090700,101,44930,26,52.43,14.14,7.90,2.68,99.25,106.80,...,1.066771,2.770382,1.011555,1.872267,3.073829,3.559012,0.0,0.269231,0.461538,4.735652
1,2023090700,194,41325,32,88.98,22.23,6.09,2.14,245.74,314.63,...,1.001798,4.189085,2.509043,1.736266,4.102659,6.671739,0.0,0.343750,0.406250,1.883215
2,2023090700,219,53591,17,75.98,10.22,3.85,2.77,268.92,343.54,...,4.906832,5.927040,0.637236,5.363283,6.049876,6.554121,0.0,0.000000,0.000000,4.906832
3,2023090700,361,38696,51,34.19,48.33,1.51,3.87,18.99,194.89,...,2.143105,6.057146,3.590496,2.667735,4.921839,9.967827,0.0,0.000000,0.352941,2.180138
4,2023090700,436,53541,20,33.67,37.80,3.59,6.08,116.90,136.35,...,2.872786,4.648680,1.019285,3.785929,4.910604,5.648982,0.0,0.000000,0.050000,2.872786


## Cleaning and Validating the Dataset

We remove rows with missing target values and inspect missingness across features to ensure data quality before modeling.

In [17]:
if "yards_gained" in model.columns:
    before = len(model)
    model = model.dropna(subset=["yards_gained"]).copy()
    print("Dropped rows missing yards_gained:", before - len(model))

missing_rate = model.isna().mean().sort_values(ascending=False)
missing_rate.head(25)

Dropped rows missing yards_gained: 0


dy_def_5                      0.011766
closev_def_5                  0.011766
angdiff_def_5                 0.011766
dx_def_5                      0.011766
dist_def_5                    0.011766
approach_def_5                0.011766
pass_location_type            0.001134
dy_def_4                      0.001063
dx_def_4                      0.001063
dist_def_4                    0.001063
approach_def_4                0.001063
closev_def_4                  0.001063
angdiff_def_4                 0.001063
team_coverage_type            0.000213
route_of_targeted_receiver    0.000213
team_coverage_man_zone        0.000213
angdiff_def_2                 0.000071
angdiff_def_3                 0.000071
frac_sep_below_2              0.000071
frac_sep_below_1              0.000071
sep_p75                       0.000071
approach_def_1                0.000071
closev_def_2                  0.000071
approach_def_2                0.000071
approach_def_3                0.000071
dtype: float64

## Saving the Modeling Dataset

The final modeling dataset is saved in both:
- Parquet format (efficient),
- CSV format (portable).

These files are used in Notebook 4.

In [18]:
out_parquet = DATA_DIR / "processed"/ "model_table_yards.parquet"
out_csv = DATA_DIR / "processed"/ "model_table_yards.csv"

model.to_parquet(out_parquet, index=False)
model.to_csv(out_csv, index=False)

print("Saved:", out_parquet)
print("Saved:", out_csv)

Saved: /Users/siamsadman/Git_Projects/git-projects/statistical_consulting_project/nfl_big_data_bowl_2026/nfl-yards-gained-prediction/data/processed/model_table_yards.parquet
Saved: /Users/siamsadman/Git_Projects/git-projects/statistical_consulting_project/nfl_big_data_bowl_2026/nfl-yards-gained-prediction/data/processed/model_table_yards.csv


## Feature Documentation (Data Dictionary)

We generate a data dictionary describing:
- each feature,
- its meaning,
- data source,
- and computation method.

This improves transparency and reproducibility.

In [19]:
rows = []

def add(col, meaning, source, how):
    if col in model.columns:
        rows.append({"column": col, "meaning": meaning, "source": source, "how_computed": how})

add("game_id", "Game identifier", "raw", "given")
add("play_id", "Play identifier", "raw", "given")
add("nfl_id_wr", "Targeted receiver player id", "raw", "player_role == Targeted Receiver")
add("ref_frame_id", "Reference frame (snapshot)", "engineered", "max frame_id for WR in input window")
add("yards_gained", "Total yards gained on play (target)", "supplementary", "given")

add("wr_x", "WR x at snapshot", "raw", "WR row at ref_frame_id")
add("wr_y", "WR y at snapshot", "raw", "WR row at ref_frame_id")
add("wr_s", "WR speed at snapshot", "raw", "WR row at ref_frame_id")
add("wr_a", "WR acceleration at snapshot", "raw", "WR row at ref_frame_id")
add("wr_dir", "WR movement direction (deg) at snapshot", "raw", "WR row at ref_frame_id")

for k in range(1, K + 1):
    add(f"dist_def_{k}", f"Euclidean distance to {k}th nearest defender at snapshot", "engineered", "rank defenders by distance to WR at ref_frame_id")
    add(f"dx_def_{k}", f"x difference WR - defender ({k}th nearest) at snapshot", "engineered", "wr_x - def_x at ref_frame_id")
    add(f"dy_def_{k}", f"y difference WR - defender ({k}th nearest) at snapshot", "engineered", "wr_y - def_y at ref_frame_id")
    add(f"angdiff_def_{k}", f"Wrapped angle difference (def_dir - wr_dir) for {k}th defender", "engineered", "wrap to [-180,180]")
    add(f"approach_def_{k}", f"Approach score for {k}th defender", "engineered", "dot(unit_v_def, vector(def->wr))/||vector||")
    add(f"closev_def_{k}", f"Projected closing speed proxy for {k}th defender", "engineered", "def_s * approach_score")

add("dist_def_mean", "Mean defender distance to WR at snapshot", "engineered", "average over all defenders at ref_frame_id")
add("dist_def_std", "Std defender distance to WR at snapshot", "engineered", "std over all defenders at ref_frame_id")
add("dist_def_p50", "Median defender distance at snapshot", "engineered", "percentile over all defenders")
add("n_def_within_3", "Number of defenders within 3 yards at snapshot", "engineered", "count(dist<=3)")
add("sep_at_throw", "Nearest-defender separation at snapshot", "engineered", "nearest defender dist at ref_frame_id")
add("sep_min", "Minimum nearest-defender separation across input frames", "engineered", "min over frames")
add("sep_mean", "Mean nearest-defender separation across input frames", "engineered", "mean over frames")
add("frac_sep_below_2", "Fraction frames with separation < 2 yards", "engineered", "mean(sep_frame<2) over frames")
add("route_of_targeted_receiver", "Route run by targeted receiver", "supplementary", "given")
add("team_coverage_man_zone", "Coverage family (Man/Zone)", "supplementary", "given")
add("down", "Down (1-4)", "supplementary", "given")
add("yards_to_go", "Yards needed for first down", "supplementary", "given")

dd = pd.DataFrame(rows).drop_duplicates("column").sort_values("column")
dd_path = DATA_DIR / "processed" / "model_table_yards_data_dictionary.csv"
dd.to_csv(dd_path, index=False)

print("Saved data dictionary:", dd_path)
dd.head(30)

Saved data dictionary: /Users/siamsadman/Git_Projects/git-projects/statistical_consulting_project/nfl_big_data_bowl_2026/nfl-yards-gained-prediction/data/processed/model_table_yards_data_dictionary.csv


,column,meaning,source,how_computed
13,angdiff_def_1,Wrapped angle difference (def_dir - wr_dir) fo...,engineered,"wrap to [-180,180]"
19,angdiff_def_2,Wrapped angle difference (def_dir - wr_dir) fo...,engineered,"wrap to [-180,180]"
25,angdiff_def_3,Wrapped angle difference (def_dir - wr_dir) fo...,engineered,"wrap to [-180,180]"
31,angdiff_def_4,Wrapped angle difference (def_dir - wr_dir) fo...,engineered,"wrap to [-180,180]"
37,angdiff_def_5,Wrapped angle difference (def_dir - wr_dir) fo...,engineered,"wrap to [-180,180]"
14,approach_def_1,Approach score for 1th defender,engineered,"dot(unit_v_def, vector(def->wr))/||vector||"
20,approach_def_2,Approach score for 2th defender,engineered,"dot(unit_v_def, vector(def->wr))/||vector||"
26,approach_def_3,Approach score for 3th defender,engineered,"dot(unit_v_def, vector(def->wr))/||vector||"
32,approach_def_4,Approach score for 4th defender,engineered,"dot(unit_v_def, vector(def->wr))/||vector||"
38,approach_def_5,Approach score for 5th defender,engineered,"dot(unit_v_def, vector(def->wr))/||vector||"


## Full Data Dictionary

In addition to the compact dictionary above, we generate a more detailed serial data dictionary containing:
- column order,
- dtype,
- missingness rate,
- explanation,
- source,
- and how each feature was computed.

In [20]:
out_path = DATA_DIR / "processed" / "model_table_yards_data_dictionary_full.csv"
cols = list(model.columns)

def _short(col: str) -> str:
    return col.replace("_", " ").strip()

def make_explainer(K: int):
    base = {
        "game_id": ("Game identifier.", "raw", "Given in the dataset."),
        "play_id": ("Play identifier within a game.", "raw", "Given in the dataset."),
        "nfl_id_wr": ("Player ID of the targeted receiver (WR).", "raw", "Selected where player_role == 'Targeted Receiver'."),
        "ref_frame_id": ("Reference frame used as the snapshot (closest moment to throw).", "engineered", "Last available input frame_id for the targeted receiver."),
        "yards_gained": ("Target: total yards gained on the play (including yards after catch).", "supplementary", "Given in supplementary_data.csv."),
        "pass_result": ("Play outcome label (e.g., C, I, IN, S, R).", "supplementary", "Given in supplementary_data.csv."),
        "wr_x": ("Targeted receiver x-position at the snapshot.", "raw", "WR x at ref_frame_id."),
        "wr_y": ("Targeted receiver y-position at the snapshot.", "raw", "WR y at ref_frame_id."),
        "wr_s": ("Targeted receiver speed at the snapshot (yards/second).", "raw", "WR s at ref_frame_id."),
        "wr_a": ("Targeted receiver acceleration at the snapshot (yards/second²).", "raw", "WR a at ref_frame_id."),
        "wr_dir": ("Targeted receiver movement direction at the snapshot (degrees).", "raw", "WR dir at ref_frame_id."),
        "wr_o": ("Targeted receiver body orientation at the snapshot (degrees).", "raw", "WR o at ref_frame_id."),
        "wr_position": ("Targeted receiver position label.", "raw", "From player_position for targeted receiver."),
        "ball_land_x": ("Ball landing x-coordinate.", "raw", "Given in tracking input file."),
        "ball_land_y": ("Ball landing y-coordinate.", "raw", "Given in tracking input file."),
        "play_direction": ("Offense play direction (left/right).", "raw", "Given in tracking input file."),
        "absolute_yardline_number": ("Field position indicator.", "raw/supplementary", "Distance from end zone for possession team."),
        "sep_at_throw": ("Nearest-defender separation at the snapshot (yards).", "engineered", "Nearest defender distance at ref_frame_id."),
        "sep_min": ("Minimum nearest-defender separation across all input frames (yards).", "engineered", "min over frames of distance to nearest defender."),
        "sep_mean": ("Mean nearest-defender separation across all input frames (yards).", "engineered", "mean over frames of distance to nearest defender."),
        "sep_std": ("Std. dev. of nearest-defender separation across input frames.", "engineered", "std over frames."),
        "sep_p25": ("25th percentile of nearest-defender separation across input frames.", "engineered", "percentile over frames."),
        "sep_p50": ("50th percentile of nearest-defender separation across input frames.", "engineered", "percentile over frames."),
        "sep_p75": ("75th percentile of nearest-defender separation across input frames.", "engineered", "percentile over frames."),
        "frac_sep_below_1": ("Fraction of input frames where separation < 1 yard.", "engineered", "mean(sep_frame < 1) over frames."),
        "frac_sep_below_2": ("Fraction of input frames where separation < 2 yards.", "engineered", "mean(sep_frame < 2) over frames."),
        "frac_sep_below_3": ("Fraction of input frames where separation < 3 yards.", "engineered", "mean(sep_frame < 3) over frames."),
        "n_frames": ("Number of input frames available for the targeted receiver.", "engineered", "count of frames for WR in input window."),
        "wr_path_length": ("Total path length the WR traveled across input frames (yards).", "engineered", "sum of step distances across frames."),
        "wr_x_start": ("WR x-position at first input frame.", "engineered", "First WR x in input frames."),
        "wr_y_start": ("WR y-position at first input frame.", "engineered", "First WR y in input frames."),
        "wr_x_end": ("WR x-position at last input frame (snapshot).", "engineered", "Last WR x in input frames."),
        "wr_y_end": ("WR y-position at last input frame (snapshot).", "engineered", "Last WR y in input frames."),
        "season": ("Season year.", "supplementary", "Given in supplementary_data.csv."),
        "week": ("Week of season.", "supplementary", "Given in supplementary_data.csv."),
        "down": ("Down (1–4).", "supplementary", "Given in supplementary_data.csv."),
        "yards_to_go": ("Yards needed for a first down.", "supplementary", "Given in supplementary_data.csv."),
        "pass_length": ("Air yards beyond LOS (can be negative).", "supplementary", "Given in supplementary_data.csv."),
        "offense_formation": ("Offensive formation.", "supplementary", "Given in supplementary_data.csv."),
        "receiver_alignment": ("Receiver alignment grouping.", "supplementary", "Given in supplementary_data.csv."),
        "route_of_targeted_receiver": ("Route type run by the targeted receiver.", "supplementary", "Given in supplementary_data.csv."),
        "play_action": ("Play action indicator (0/1).", "supplementary", "Given in supplementary_data.csv."),
        "dropback_type": ("QB dropback type.", "supplementary", "Given in supplementary_data.csv."),
        "dropback_distance": ("QB dropback distance (yards).", "supplementary", "Given in supplementary_data.csv."),
        "pass_location_type": ("QB location at throw (inside/outside).", "supplementary", "Given in supplementary_data.csv."),
        "defenders_in_the_box": ("Number of defenders near LOS.", "supplementary", "Given in supplementary_data.csv."),
        "team_coverage_man_zone": ("Coverage family: Man vs Zone.", "supplementary", "Given in supplementary_data.csv."),
        "team_coverage_type": ("Specific coverage type label.", "supplementary", "Given in supplementary_data.csv."),
        "pre_snap_home_team_win_probability": ("Home win probability before play.", "supplementary", "Given in supplementary_data.csv."),
        "pre_snap_visitor_team_win_probability": ("Visitor win probability before play.", "supplementary", "Given in supplementary_data.csv."),
        "expected_points": ("Expected points on the play.", "supplementary", "Given in supplementary_data.csv."),
        "expected_points_added": ("Delta EP on the play.", "supplementary", "Given in supplementary_data.csv."),
    }

    def explain_pattern(col: str):
        for prefix, meaning, how in [
            ("dist_def_", "Euclidean distance from WR to the {k}th nearest defender at snapshot (yards).", "Rank defenders by distance at ref_frame_id; dist = sqrt(dx^2 + dy^2)."),
            ("dx_def_", "Signed x-difference (WR − defender) for the {k}th nearest defender at snapshot (yards).", "dx = wr_x − def_x at ref_frame_id."),
            ("dy_def_", "Signed y-difference (WR − defender) for the {k}th nearest defender at snapshot (yards).", "dy = wr_y − def_y at ref_frame_id."),
            ("angdiff_def_", "Wrapped angle difference (def_dir − wr_dir) for the {k}th nearest defender (degrees).", "wrap((def_dir − wr_dir)) into [-180, 180]."),
            ("approach_def_", "Approach score for {k}th nearest defender: +1 moving toward WR, −1 moving away.", "Dot(unit_v_def, vector(def→WR)) / ||vector|| at ref_frame_id."),
            ("closev_def_", "Projected closing-speed proxy toward WR for {k}th defender (yards/second).", "closev = def_s × approach_score."),
        ]:
            if col.startswith(prefix):
                try:
                    k = int(col.replace(prefix, ""))
                except Exception:
                    return None
                if 1 <= k <= K:
                    return (meaning.format(k=k), "engineered", how)
                return None

        if col.startswith("dist_def_") and col in {"dist_def_min", "dist_def_mean", "dist_def_std", "dist_def_p10", "dist_def_p25", "dist_def_p50", "dist_def_p75", "dist_def_p90"}:
            m = {
                "dist_def_min": ("Minimum distance from WR to any defender at snapshot.", "engineered", "min over defenders of Euclidean distance at ref_frame_id."),
                "dist_def_mean": ("Mean distance from WR to defenders at snapshot.", "engineered", "mean over defenders at ref_frame_id."),
                "dist_def_std": ("Std. dev. of defender distances at snapshot.", "engineered", "std over defenders at ref_frame_id."),
                "dist_def_p10": ("10th percentile of defender distances at snapshot.", "engineered", "percentile over defenders at ref_frame_id."),
                "dist_def_p25": ("25th percentile of defender distances at snapshot.", "engineered", "percentile over defenders at ref_frame_id."),
                "dist_def_p50": ("Median defender distance at snapshot.", "engineered", "percentile over defenders at ref_frame_id."),
                "dist_def_p75": ("75th percentile defender distance at snapshot.", "engineered", "percentile over defenders at ref_frame_id."),
                "dist_def_p90": ("90th percentile defender distance at snapshot.", "engineered", "percentile over defenders at ref_frame_id."),
            }
            return m.get(col, None)

        if col.startswith("n_def_within_"):
            try:
                r = int(col.replace("n_def_within_", ""))
                return (f"Number of defenders within {r} yards of WR at snapshot.", "engineered", f"count(dist <= {r}) over defenders at ref_frame_id.")
            except Exception:
                return None

        for p in ["wr_s_", "wr_a_", "wr_dir_"]:
            if col.startswith(p) and col.split("_")[-1] in {"mean", "max", "std", "min"}:
                measure = p.replace("wr_", "").replace("_", "")
                stat = col.split("_")[-1]
                if measure == "s":
                    return (f"WR speed {stat} across input frames.", "engineered", f"{stat} of WR speed over input frames.")
                if measure == "a":
                    return (f"WR acceleration {stat} across input frames.", "engineered", f"{stat} of WR acceleration over input frames.")
                if measure == "dir":
                    return (f"WR direction {stat} across input frames.", "engineered", f"{stat} of WR direction over input frames (degrees).")

        return None

    return base, explain_pattern

base_map, pattern_fn = make_explainer(K)

records = []
for i, c in enumerate(cols, start=1):
    if c in base_map:
        meaning, source, how = base_map[c]
    else:
        pat = pattern_fn(c)
        if pat is not None:
            meaning, source, how = pat
        else:
            meaning = f"{_short(c)}."
            if c in {"yards_gained"}:
                source = "supplementary"
                how = "Given."
            elif c.startswith(("wr_", "sep_", "dist_def_", "dx_def_", "dy_def_", "angdiff_", "approach_", "closev_", "n_def_within_")):
                source = "engineered"
                how = "Computed in feature engineering pipeline."
            else:
                source = "raw/unknown"
                how = "Carried through or computed; verify in pipeline."

    dtype = str(model[c].dtype)
    missing_pct = float(model[c].isna().mean() * 100.0)

    records.append({
        "order": i,
        "column": c,
        "dtype": dtype,
        "missing_pct": round(missing_pct, 3),
        "explanation": meaning,
        "source": source,
        "how_computed": how,
    })

dd_full = pd.DataFrame(records)
dd_full.to_csv(out_path, index=False)

print("Saved full data dictionary:", out_path)
dd_full.head(30)

Saved full data dictionary: /Users/siamsadman/Git_Projects/git-projects/statistical_consulting_project/nfl_big_data_bowl_2026/nfl-yards-gained-prediction/data/processed/model_table_yards_data_dictionary_full.csv


,order,column,dtype,missing_pct,explanation,source,how_computed
0,1,game_id,int64,0.000,Game identifier.,raw,Given in the dataset.
1,2,play_id,int64,0.000,Play identifier within a game.,raw,Given in the dataset.
2,3,nfl_id_wr,int64,0.000,Player ID of the targeted receiver (WR).,raw,Selected where player_role == 'Targeted Receiv...
3,4,ref_frame_id,int64,0.000,Reference frame used as the snapshot (closest ...,engineered,Last available input frame_id for the targeted...
4,5,wr_x,float64,0.000,Targeted receiver x-position at the snapshot.,raw,WR x at ref_frame_id.
5,6,wr_y,float64,0.000,Targeted receiver y-position at the snapshot.,raw,WR y at ref_frame_id.
6,7,wr_s,float64,0.000,Targeted receiver speed at the snapshot (yards...,raw,WR s at ref_frame_id.
7,8,wr_a,float64,0.000,Targeted receiver acceleration at the snapshot...,raw,WR a at ref_frame_id.
8,9,wr_dir,float64,0.000,Targeted receiver movement direction at the sn...,raw,WR dir at ref_frame_id.
9,10,wr_o,float64,0.000,Targeted receiver body orientation at the snap...,raw,WR o at ref_frame_id.


## Summary

This notebook produced a rich, leakage-free modeling dataset that:
- incorporates spatial, temporal, and contextual information,
- reflects defensive structure and pursuit behavior,
- follows best practices for predictive modeling.

The dataset is now ready for regression modeling in Notebook 4.